# 02 · METAR y variables base (137)

v30 parte de 137 variables heredadas de V21–V27: relojes NM y del plan de vuelo, coherencia entre relojes, tráfico y
colas, contexto de congestión, METAR en el despegue y en el pushback, retención en puerta, calendario y categóricas.
Las construye `prc/base` (procesamiento → tablas → matriz) a través de `scripts/variables_base.py`.

Los METAR son partes del IEM (Iowa State) de los diez aeropuertos, que `scripts/descarga_metar.py <año>` deja en
`Data/metar/2025/` y `Data/metar/2026/` (un fichero de texto por aeropuerto y mes). Por defecto reutilizamos las
tablas de `Data/interim/` y `Data/features/base/` con las que se entrenó v30; sus METAR de 2025 son de una descarga
anterior.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT)]
from cuaderno import elegir_periodo, existen, interruptor, run  # noqa: E402

PERIODO = elegir_periodo("ranking")   # abierto suelto: "ranking" (ene + jul 2026) o "final" (ene, feb, jun y jul 2026)
FINAL = PERIODO == "final"
import periodo as PER  # noqa: E402  rutas del periodo: PER.RANKING (fichero a predecir), PER.FEAT (variables)

DESCARGAR_METAR = interruptor("DESCARGAR_METAR", False)
REHACER = interruptor("REHACER_BASE", False)

In [ ]:
if DESCARGAR_METAR:
    run("scripts/descarga_metar.py", "2025")
    run("scripts/descarga_metar.py", "2026")
else:
    print("METAR: no se descargan (DESCARGAR_METAR = 0)")

In [ ]:
if FINAL:   # solo 2026, en Data/interim_final y Data/features_final/base; las tablas de 2025 son las mismas
    run("scripts/preparar_final.py", "base", *(["--rehacer"] if REHACER else []))
else:
    run("scripts/variables_base.py", *(["--rehacer"] if REHACER else []))

Comprobamos que la matriz base tiene 137 variables, las mismas en 2025 y 2026, y que 2026 tiene las filas de la
plantilla.

In [ ]:
import matriz as MZ
from prc.base import config as K

d25, X25, ref25, cats = MZ.cargar(2025, [], "")
d26, X26, ref26, _ = MZ.cargar(2026, [], "")
assert X25.shape[1] == X26.shape[1] == 137 and list(X25.columns) == list(X26.columns)
assert len(d26) == K.N_FILAS_ENVIO
print(f"2025: {X25.shape[0]:,} salidas · 2026: {X26.shape[0]:,} salidas · {X25.shape[1]} variables ({len(cats)} categóricas)")

### Referencias guardadas, recalculadas desde 2025

v30 lee tres tablas ajustadas con 2025 y guardadas con la entrega de V26/V27 (no usa ningún modelo ni predicción de
V27). Aquí las recalculamos y las comparamos con las guardadas: la tabla P10 y el vocabulario de categóricas salen
idénticos, y en la tabla de recuperación de vuelos solo cambian los empates (la guardada los resolvió al azar; el
código actual, por orden alfabético).

In [ ]:
import pandas as pd
from prc.base import config as K, matriz as Mo, tablas as F

nuevas = Mo.ajustar_referencias(Mo.cargar(2025), Mo.receta())
guardadas = Mo.leer_referencias()
orden = lambda t: t.sort_values(["nivel", "clave"]).reset_index(drop=True)
print("tabla P10 idéntica:", orden(nuevas.p10).equals(orden(guardadas.p10)), f"({len(nuevas.p10):,} grupos)")
print("vocabulario de categóricas idéntico:", nuevas.categorias == guardadas.categorias)
rec_n = F.tabla_recuperacion(K.interim("dep", 2025))
rec_g = pd.read_parquet(F.TABLA_RECUPERACION)
m = rec_n.merge(rec_g, on=["airport", "FLIGHT_mvt"], suffixes=("_n", "_g"))
distintas = sum(int((m[c + "_n"].fillna("") != m[c + "_g"].fillna("")).sum()) for c in ("op", "seg", "wk"))
print(f"tabla de recuperación de vuelos: {distintas} de {len(rec_g):,} entradas distintas (empates)")